# Runtime handles

The SDK gives you a server-backed runtime. **`AsyncWorkflowRuntime`**
(async-first) and **`WorkflowRuntime`** (sync) offer a
`WorkflowRuntime.from_config(...)` surface — the only difference from an
in-process runtime is a `client=` argument, because execution happens on the
server:

| Concept | SDK (server-side) |
|---|---|
| Build a runtime | `runtime = await AsyncWorkflowRuntime.from_config(cfg, client=client)` |
| Drive a turn | `async for e in runtime.arun(input): ...` |

`AsyncWorkflowRuntime` / `WorkflowRuntime` subclass the lower-level
`AsyncWorkflowHandle` / `WorkflowHandle`; the `aupload_and_get_handle`
factory remains available as a back-compatible alias.

All runtime state (turn count, message history, node state) lives on the
server; the runtime only persists the `run_id` echoed back from the first
turn so subsequent turns resume the same session.

See the companion guide: [`../docs/runtime.md`](../docs/runtime.md).


> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from langchain_core.messages import HumanMessage

from interactly import AsyncWorkflowClient, AsyncWorkflowRuntime, aupload_and_get_handle
from interactly.configs import (
    ConditionConfig,
    ConditionalEdgeConfig,
    DirectEdgeConfig,
    LLMNodeRunInput,
    NodesRunInputs,
    OPENAIModel,
    OpenAILLMConfig,
    PromptConfig,
    SayLLMNodeConfig,
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    WorkflowRunInput,
)
from interactly.runtime.events import (
    AssistantResponseEvent,
    BusyWaitForUserMessageEvent,
)

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## Build a small workflow config

Same shape as [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb): a start
`SayLLMNodeConfig`, a self-looping assistant, and a static end node,
assembled into a `WorkflowConfigFullyHydrated`.

In [ ]:
greeting = SayLLMNodeConfig(
    name="Greeting",
    is_start=True,
    self_loop=False,
    wait_for_user_message=False,
    main_response_config=PromptConfig(prompt="Greet the user in under 15 words and ask how you can help."),
    llms_config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_NANO, max_tokens=100),
)
assistant = SayLLMNodeConfig(
    name="Assistant",
    self_loop=True,
    wait_for_user_message=True,
    main_response_config=PromptConfig(prompt="You are a helpful, concise assistant."),
    llms_config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4, max_tokens=300, temperature=0.2),
)
end = SayStaticMessageNodeConfig(
    name="End",
    static_messages_config=StaticMessagesConfig(static_messages=["Thanks for chatting. Goodbye!"]),
)

config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(name="Handle Demo", category="System Examples"),
    node_configs=[greeting, assistant, end],
    edge_configs=[
        DirectEdgeConfig(
            source_node_logical_id=greeting.logical_id,
            destination_node_logical_id=assistant.logical_id,
            name="to assistant",
        ),
        ConditionalEdgeConfig(
            source_node_logical_id=assistant.logical_id,
            destination_node_logical_id=end.logical_id,
            name="end conversation",
            condition=ConditionConfig(condition_freeform="The user says goodbye or is done."),
        ),
    ],
)

## Create a runtime from the config

`await AsyncWorkflowRuntime.from_config(config, client=client, ...)` creates
the workflow on the server (`client.workflows.create_from_config`) and hands
back an `AsyncWorkflowRuntime` bound to its id. `dynamic_variables` become
defaults merged underneath each turn's own `dynamic_variables`. (The sync twin
is `WorkflowRuntime.from_config(config, client=sync_client, ...)`; the
`aupload_and_get_handle(client, config, ...)` factory is an equivalent alias.)

In [ ]:
runtime = await AsyncWorkflowRuntime.from_config(
    config,
    client=client,
    name="Handle Demo",
    dynamic_variables={},  # defaults merged into every turn
)
print(runtime.workflow_id)
print(runtime.run_id)  # None until the first arun() call

## Run a turn

Each turn you build a `WorkflowRunInput` and iterate `runtime.arun(...)`,
which yields typed events. The input matches
`wf_examples/wf_example_progression_1.py`:

- `thread_to_node_inputs` maps a thread id (string, `"0"` for the main
  thread) to a `NodesRunInputs`.
- `NodesRunInputs.node_run_inputs` holds per-node inputs; here a single
  `LLMNodeRunInput` carrying the user's `messages`.

On the first call `run_id` is unset so the server creates a new session;
the runtime records the returned `run_id` and echoes it on later turns.

In [ ]:
async def turn(text):
    run_input = WorkflowRunInput(
        thread_to_node_inputs={"0": NodesRunInputs(
            node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content=text)])]
        )},
        dynamic_variables={},
    )
    async for event in runtime.arun(run_input):
        if isinstance(event, AssistantResponseEvent):
            print(f"ASSISTANT ({event.origin_node_name}): {event.content}")
        elif isinstance(event, BusyWaitForUserMessageEvent):
            print(f"[waiting for user at {event.origin_node_name}]")

await turn("Hi, what does a copay mean?")
print("run_id after turn 1:", runtime.run_id)

# Second turn resumes the SAME session (same run_id echoed automatically).
await turn("Thanks, that is all. Goodbye!")

## Sessions, `run_id`, and `reset()`

- `runtime.run_id` is the server session id — `None` before the first
  turn, then stable across turns.
- The async `AsyncWorkflowRuntime` serialises overlapping `arun()` calls
  with a per-runtime lock; start a second concurrent conversation with a
  separate runtime.
- `runtime.reset()` forgets the current `run_id` so the **next** `arun()`
  begins a fresh session against the same workflow.

In [ ]:
runtime.reset()
print(runtime.run_id)  # back to None -> next turn starts a new session
await turn("Hello again from a brand new session!")

## The typed event catalog

`runtime.arun()` yields parsed event objects (via `parse_event`) rather
than raw dicts, so you can `isinstance`-check them. They are re-exported
from `interactly.runtime.events`. A sampling:

| Event | Meaning |
|---|---|
| `AssistantResponseEvent` | assistant produced text (`.content`) |
| `UserMessagesEvent` | the user's messages entered a node |
| `BusyWaitForUserMessageEvent` | node is waiting for user input |
| `StartWorkflowEvent` / `EndWorkflowEvent` | workflow lifecycle |
| `StartConversationNodeEvent` / `EndConversationNodeEvent` | node lifecycle |
| `DirectEdgeEvent` / `ConditionalEdgeEvent` / `SelfLoopEdgeEvent` | edge traversal |
| `HttpRequestNodeEvent` / `SendSMSNodeEvent` / `AthenaNodeEvent` | side-effecting nodes |
| `WorkflowErrorEvent` / `WorkflowWarningEvent` / `WorkflowDebugLogEvent` | diagnostics |

`BaseEvent` is the common ancestor. Import only what you match on.

In [ ]:
from interactly.runtime.events import (
    BaseEvent,
    StartWorkflowEvent,
    EndWorkflowEvent,
    UserMessagesEvent,
    WorkflowErrorEvent,
)

# Inspect every event type on one turn instead of only the two we handled.
run_input = WorkflowRunInput(
    thread_to_node_inputs={"0": NodesRunInputs(
        node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content="What is a premium?")])]
    )},
    dynamic_variables={},
)
async for event in runtime.arun(run_input):
    assert isinstance(event, BaseEvent)
    print(type(event).__name__)

## Cleanup

Delete the demo workflow and close the client.

In [ ]:
await client.workflows.delete(runtime.workflow_id)
await client.close()
print("cleaned up")

## See also

- Guide: [`../docs/runtime.md`](../docs/runtime.md)
- Full example: [`../wf_examples/wf_example_progression_1.py`](../wf_examples/wf_example_progression_1.py)
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — build & chat with a workflow via a handle